# Predicting day of year from local weather

Compare air temperature alone (`tas`) with air temperature and dew point
(`tas + tdew`). For each input set and history length, fit the same pooled
ridge probe directly to district-level weather histories. The probe estimates
the sine and cosine of the annual phase.

The error summaries use the saved validation dates. The Cartesian and polar
plots pool training and validation dates across years for windows of 1, 6, 10,
20, and 30 days. Both use the same 366-day calendar, with February 29
reserved for leap years.

In [ ]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import sys

cwd = Path.cwd()
repo = next(
    (
        path for path in (cwd, *cwd.parents)
        if (path / "configs" / "training.example.yaml").is_file()
    ),
    None,
)
if repo is None:
    raise FileNotFoundError("Open the notebook from the sari-weather-mortality checkout.")

sys.path.insert(0, str(repo))
sys.path.insert(0, str(repo / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Load helpers from this checkout even if Jupyter cached another notebooks package.
utils_path = repo / "notebooks" / "utils.py"
spec = spec_from_file_location("sari_notebook_utils", utils_path)
notebook_utils = module_from_spec(spec)
spec.loader.exec_module(notebook_utils)


from calendar import month_abbr

from matplotlib.colors import PowerNorm
from matplotlib.ticker import PercentFormatter
import torch

from sari_t.data.weather import load_weather
from sari_t.run.checkpoint import load_run_config

## Select weather inputs and analysis dates

In [ ]:
groups = {
    "tas": ("temperature_kernel_weather_only", ("tas",)),
    "tas + tdew": ("temperature_kernel_weather_only_tas_tdew", ("tas", "tdew")),
}
runs = {}
for feature, (group, channels) in groups.items():
    feature_runs = {}
    for entry in sorted((repo / "runtime" / "training" / "sets" / group).iterdir()):
        run = entry.resolve()
        saved = load_run_config(run / "config.yaml")
        lag = saved.model_config.kernel_T_days
        if (lag in feature_runs or saved.selected_weather != channels
                or saved.model_config.include_sari or saved.model_config.include_covid):
            raise ValueError(f"Unexpected weather-only configuration in {run}")
        feature_runs[lag] = run
    if set(feature_runs) != set(range(1, 31)):
        raise ValueError(f"Incomplete lag sweep for {feature}")
    runs[feature] = feature_runs

## Fit and score the probes

Run the fitting cells once. Both plot types reuse the counts in memory.

In [ ]:
config = load_run_config(runs["tas"][9] / "config.yaml")
period = config.period
dates = pd.date_range(period.start_date, period.end_date)
weeks = len(dates) // 7
fraction = config.training_config.validation_fraction
validation_days = 7 * int(weeks * fraction)
train_days = len(dates) - validation_days
if len(dates) % 7 or not 0 < validation_days < len(dates):
    raise ValueError("Expected complete training and validation weeks.")
validation_start = dates[train_days]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
year_lengths = np.where(dates.is_leap_year, 366, 365)
day_of_year = dates.dayofyear.to_numpy() - 1
phase = 2 * np.pi * day_of_year / year_lengths
target = torch.as_tensor(np.column_stack((np.cos(phase), np.sin(phase))),
                         device=device, dtype=torch.float64)
train = torch.as_tensor(dates < validation_start, device=device)
test = ~train
days_in_year = torch.as_tensor(year_lengths, device=device)
calendar_columns = torch.as_tensor(day_of_year + ((year_lengths == 365) & (day_of_year >= 59)),
                                  device=device)
print(f"Training {period.start_date} to {dates[train_days - 1].date()}")
print(f"Validation {validation_start.date()} to {period.end_date}")

In [ ]:
def pooled_probe(batches, *, collect_heatmap=False):
    """Fit one standardized ridge decoder with bounded memory, then score each district."""
    target_fit = target[train]
    intercept = target_fit.mean(dim=0)
    centered_target = target_fit - intercept
    count = 0
    for batch in batches():
        batch = batch.to(device=device, dtype=target.dtype)
        fit = batch[:, train].reshape(batch.shape[0] * train_days, -1)
        if count == 0:
            width = fit.shape[1]
            feature_sum = fit.new_zeros(width)
            cross_features = fit.new_zeros((width, width))
            cross_target = fit.new_zeros((width, 2))
        feature_sum += fit.sum(dim=0)
        cross_features += fit.T @ fit
        cross_target += fit.T @ centered_target.repeat(batch.shape[0], 1)
        count += len(fit)

    center = feature_sum / count
    covariance = cross_features - count * torch.outer(center, center)
    scale = (covariance.diag() / count).clamp_min(0).sqrt()
    scale = torch.where(scale == 0, 1.0, scale)
    standardized = covariance / torch.outer(scale, scale)
    penalty = 0.01 * count * torch.eye(width, device=device, dtype=target.dtype)
    weights = torch.linalg.solve(standardized + penalty, cross_target / scale[:, None])
    actual = torch.atan2(target[test, 1], target[test, 0])
    errors = []
    daily_errors = []
    if collect_heatmap:
        counts = torch.zeros(366 * 366, device=device, dtype=torch.long)

    for batch in batches():
        batch = batch.to(device=device, dtype=target.dtype)
        held = batch[:, test].reshape(batch.shape[0], validation_days, width)
        predicted = ((held - center) / scale) @ weights + intercept
        angle = torch.atan2(predicted[..., 1], predicted[..., 0])
        difference = (angle - actual + torch.pi) % (2 * torch.pi) - torch.pi
        if collect_heatmap:
            all_days = batch.reshape(batch.shape[0], len(dates), width)
            all_predictions = ((all_days - center) / scale) @ weights + intercept
            all_angles = torch.atan2(all_predictions[..., 1], all_predictions[..., 0])
            phase_fraction = all_angles.remainder(2 * torch.pi) / (2 * torch.pi)
            estimated = (phase_fraction * days_in_year).round().long() % days_in_year
            estimated = estimated + ((days_in_year == 365) & (estimated >= 59))
            indices = estimated * 366 + calendar_columns
            counts += torch.bincount(indices.flatten(), minlength=counts.numel())
        absolute_days = difference.abs() * 365.25 / (2 * torch.pi)
        errors.extend(torch.quantile(absolute_days, 0.5, dim=1).cpu().tolist())
        daily_errors.append(absolute_days)
    daily_errors = torch.cat(daily_errors)
    seasonal = {}
    for season, mask in {**season_masks, **month_masks}.items():
        values = daily_errors[:, mask].flatten()
        if values.numel():
            q1, median, q3 = torch.quantile(values, values.new_tensor([0.25, 0.5, 0.75])).cpu().tolist()
            seasonal[season] = (values.numel(), median, q1, q3, q3 - q1)
    heatmap = counts.reshape(366, 366).cpu() if collect_heatmap else None
    return errors, seasonal, heatmap

In [ ]:
validation_months = dates[dates >= validation_start].month
season_masks = {
    season: torch.as_tensor(validation_months.isin(months), device=device)
    for season, months in {
        "Winter": (12, 1, 2), "Spring": (3, 4, 5),
        "Summer": (6, 7, 8), "Autumn": (9, 10, 11),
    }.items()
}

month_masks = {
    month_abbr[month]: torch.as_tensor(validation_months == month, device=device)
    for month in range(1, 13)
}

heatmap_lags = (1, 6, 10, 20, 30)
heatmaps = {}
rows = []
seasonal_rows = []
for family, family_runs in runs.items():
    print(f"Fitting probes for {family}", flush=True)
    first = load_run_config(next(iter(family_runs.values())) / "config.yaml")
    weather = load_weather(first.data_config, first.selected_weather)
    first_day = (period.start_date - weather.start_date).days
    end = first_day + len(dates)
    if end > weather.values.shape[-1] or first_day < max(family_runs) - 1:
        raise ValueError(f"Missing weather dates or lag history for {family}")
    for lag, run in sorted(family_runs.items()):
        saved = load_run_config(run / "config.yaml")
        if (saved.period != period or saved.training_config.validation_fraction != fraction
                or saved.selected_weather != first.selected_weather or saved.data_config != first.data_config):
            raise ValueError(f"Inconsistent dates, split, or data in {run}")
        start = first_day - lag + 1
        window = weather.values[..., start:end]
        def weather_batches():
            for batch in window.split(16):
                yield batch.unfold(-1, lag, 1).transpose(1, 2)

        errors, seasonal, heatmap = pooled_probe(
            weather_batches, collect_heatmap=lag in heatmap_lags,
        )
        if heatmap is not None:
            heatmaps[family, lag] = heatmap
        for season, stats in seasonal.items():
            seasonal_rows.append((family, lag, season, *stats))
        for district, error in enumerate(errors):
            rows.append((family, lag, district, error))

## Validation errors

In [ ]:
metric = "Median date error (days)"
results = pd.DataFrame(rows, columns=["Inputs", "Window (days)", "District index", metric])
summary = results.groupby(["Inputs", "Window (days)"])[metric].median()
display(summary.unstack("Inputs").round(2))

seasonal_errors = pd.DataFrame(seasonal_rows, columns=[
    "Inputs", "Window (days)", "Season", "District-days",
    metric, "Q1 (days)", "Q3 (days)", "IQR (days)",
]).set_index(["Inputs", "Window (days)", "Season"])
monthly_errors = seasonal_errors[seasonal_errors.index.get_level_values("Season").isin(month_masks)]
monthly_errors = monthly_errors.rename_axis(index={"Season": "Month"})
seasonal_errors = seasonal_errors[seasonal_errors.index.get_level_values("Season").isin(season_masks)]
display(seasonal_errors.round(2))

colors = {"tas": "#0072B2", "tas + tdew": "#D55E00"}

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for feature, color in colors.items():
    values = summary.loc[feature]
    ax.plot(values.index, values, label=feature, color=color)
ax.set(title="Overall validation", xlabel="Weather input window (days)",
       ylabel="Median district date error (days)")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

In [ ]:
for season in season_masks:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for feature, color in colors.items():
        subset = seasonal_errors.xs((feature, season), level=("Inputs", "Season"))
        values = subset.groupby(level="Window (days)").median(numeric_only=True)
        ax.plot(values.index, values[metric], color=color, label=feature)
        ax.fill_between(values.index, values["Q1 (days)"], values["Q3 (days)"], color=color, alpha=0.10)
    ax.set(title=f"{season} validation", xlabel="Weather input window (days)", ylabel="Median date error (days)")
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()

fig, axes = plt.subplots(4, 3, figsize=(13, 11), sharex=True, sharey=True, layout="constrained")
for month, ax in zip(month_masks, axes.flat):
    for feature, color in colors.items():
        subset = monthly_errors.xs((feature, month), level=("Inputs", "Month"))
        values = subset.groupby(level="Window (days)").median(numeric_only=True)
        ax.plot(values.index, values[metric], color=color, label=feature)
        ax.fill_between(values.index, values["Q1 (days)"], values["Q3 (days)"], color=color, alpha=0.10)
    ax.set_title(month)
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside upper center", ncol=2, frameon=False)
fig.supxlabel("Weather input window (days)")
fig.supylabel("Median validation date error (days)")
plt.show()

## Date predictions and circular errors

Each input set has a Cartesian–polar pair. In the Cartesian panel, the orange diagonal marks correct predictions. In the polar panel, angle gives the true date clockwise from January 1. The orange ring marks zero error; estimates inside are early and estimates outside are late. Errors wrap at half a year. All panels share one colour scale.

In [ ]:
shares = {key: (counts / counts.sum(dim=0).clamp_min(1)).numpy()
          for key, counts in heatmaps.items()}
positive = np.concatenate([values[values > 0] for values in shares.values()])
norm = PowerNorm(gamma=0.5, vmin=0, vmax=np.quantile(positive, 0.995))

year_days = 366
half_year = year_days // 2
true_days = np.arange(year_days)
estimated_rows = (np.arange(-half_year, half_year)[:, None] + true_days[None, :]) % year_days
day_edges = np.arange(year_days + 1) + 0.5
angle_edges = (np.arange(year_days + 1) - 0.5) * 2 * np.pi / year_days
radius_edges = np.arange(year_days + 1)
ring_angles = np.linspace(0, 2 * np.pi, year_days + 1)
month_starts = pd.date_range("2000-01-01", periods=12, freq="MS")[::3]
tick_days = np.array([-120, -60, 0, 60, 120, 182])

fig = plt.figure(figsize=(15, 15), layout="constrained")
grid = fig.add_gridspec(len(heatmap_lags), 2 * len(colors))
axes = []
for row, lag in enumerate(heatmap_lags):
    for pair, feature in enumerate(colors):
        values = shares[feature, lag]
        xy = fig.add_subplot(grid[row, 2 * pair])
        polar = fig.add_subplot(grid[row, 2 * pair + 1], projection="polar")
        axes.extend((xy, polar))

        xy.pcolormesh(day_edges, day_edges, values, cmap="Blues", norm=norm)
        xy.plot([1, year_days], [1, year_days], color="#D55E00", linewidth=0.9)
        xy.set_aspect("equal")
        if pair == 0:
            xy.set_ylabel(f"{lag}-day window\nPredicted day")
        if row == len(heatmap_lags) - 1:
            xy.set_xlabel("True day of year")

        polar.grid(False)
        mesh = polar.pcolormesh(angle_edges, radius_edges, values[estimated_rows, true_days],
                               cmap="Blues", norm=norm)
        polar.plot(ring_angles, np.full_like(ring_angles, half_year + 0.5),
                   color="#D55E00", linewidth=1)
        polar.set_theta_zero_location("N")
        polar.set_theta_direction(-1)
        polar.set_xticks((month_starts.dayofyear - 1) * 2 * np.pi / year_days,
                         month_starts.strftime("%b"))
        polar.set_rticks(tick_days + half_year + 0.5,
                         [f"{day:+d}d" if day else "0d" for day in tick_days])
        polar.set_rlabel_position(40)
        polar.set_ylim(0, year_days)
        polar.grid(alpha=0.2)

        if row == 0:
            xy.set_title(f"{feature} · Cartesian")
            polar.set_title(f"{feature} · Polar", pad=14)

fig.colorbar(mesh, ax=axes, extend="max", format=PercentFormatter(1),
             label="Share of district predictions")
plt.show()